In [ ]:
from pynq import Overlay, allocate
from pynq.lib.video import *
import numpy as np
import PIL.Image

ol = Overlay("./base.bit")

frame_shape = (720, 1280)
dma_send = ol.axi_dma_0.sendchannel
dma_recv = ol.axi_dma_0.recvchannel
input_buffer = allocate(shape=frame_shape, dtype=np.uint32)
output_buffer = allocate(shape=frame_shape, dtype=np.uint32)

In [ ]:
mipi = ol.mipi
videomode = VideoMode(frame_shape[1], frame_shape[0], 24)
mipi.configure(videomode)
mipi.start()

In [ ]:
import cv2

frame = mipi.readframe()
input_buffer[:] = (frame[:,:,2].astype(np.uint32) << 16 |
                   frame[:,:,1].astype(np.uint32) << 8 |
                   frame[:,:,0].astype(np.uint32)
                  )
dma_send.transfer(input_buffer)
dma_recv.transfer(output_buffer)
dma_send.wait()
dma_recv.wait()
image = np.empty((frame_shape[0], frame_shape[1], 3), dtype=np.uint8)
image[:,:,0] = (output_buffer >> 16).astype(np.uint8)
image[:,:,1] = (output_buffer >> 8).astype(np.uint8)
image[:,:,2] = (output_buffer).astype(np.uint8)

PIL.Image.fromarray(image)

In [ ]:
displayport = DisplayPort()
displayport.configure(VideoMode(1920, 1080, 24), PIXEL_RGB)

from time import time

for _ in range (3000):
    dp_frame_buffer = displayport.newframe()
    frame = mipi.readframe()
    input_buffer[:] = (frame[:,:,2].astype(np.uint32) << 16 |
                       frame[:,:,1].astype(np.uint32) << 8 |
                       frame[:,:,0].astype(np.uint32)
                      )
    
    dma_send.transfer(input_buffer)
    dma_recv.transfer(output_buffer)
    dma_send.wait()
    dma_recv.wait()

    buf = np.array(output_buffer, copy=True)
    dp_frame_buffer[:frame_shape[0],:frame_shape[1], 2] = (buf >> 16)
    dp_frame_buffer[:frame_shape[0],:frame_shape[1], 1] = (buf >> 8)
    dp_frame_buffer[:frame_shape[0],:frame_shape[1], 0] = (buf)

    displayport.writeframe(dp_frame_buffer)


In [ ]:
displayport.close()
mipi.stop()